# Kitoboy PIE Evaluation

Ноутбук для прогона сконвертированного span-датасета через локальный `kitoboy-pie` и подсчета span-level метрик.

Запускать из папки `synthetic_data_generation/synthesizer_agent/notebooks`.

Ожидается, что рядом на рабочем столе есть клон `kitoboy-pie` с API, возвращающим spans: `start`, `end`, `label`, `text`.

In [3]:
from pathlib import Path
import json
import sys
from collections import Counter, defaultdict

import pandas as pd
from fastapi.testclient import TestClient

DATASET_PATH = Path("../outputs/converted_synthesized_pii_spans.jsonl").resolve()
KITBOY_PIE_DIR = Path("../../../../kitoboy-pie").resolve()

EXCLUDED_LABELS = {"PASSPORT_RF"}
BATCH_SIZE = 32

print(f"Dataset:     {DATASET_PATH}")
print(f"Kitoboy PIE: {KITBOY_PIE_DIR}")

Dataset:     /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/converted_synthesized_pii_spans.jsonl
Kitoboy PIE: /Users/artemzmailov/Desktop/kitoboy-pie


## Load Dataset

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            item["_jsonl_line"] = line_no
            rows.append(item)
    return rows

rows = load_jsonl(DATASET_PATH)
print(f"Loaded rows: {len(rows)}")

label_counts = Counter(
    ent["label"]
    for item in rows
    for ent in item.get("entities", [])
)
print(dict(sorted(label_counts.items())))

Loaded rows: 1825
{'ADDRESS': 305, 'BANK_CARD': 310, 'EMAIL': 310, 'NAME': 309, 'ORGANIZATION': 309, 'PASSPORT_RF': 310, 'PHONE_NUMBER': 309, 'TELEGRAM': 308, 'VK': 309}


## Import Local Kitoboy PIE

Импортируем локальный клон как Python-модуль. При импорте `app.main` загрузятся Navec/Slovnet модели, поэтому первая ячейка может быть не мгновенной.

In [3]:
import os

if str(KITBOY_PIE_DIR) not in sys.path:
    sys.path.insert(0, str(KITBOY_PIE_DIR))

# app.main initializes models with relative paths like data/navec_...,
# so import it while cwd points to the kitoboy-pie repository.
notebook_cwd = Path.cwd()
os.chdir(KITBOY_PIE_DIR)
try:
    from app.main import app
finally:
    os.chdir(notebook_cwd)

client = TestClient(app)
print("Kitoboy PIE app loaded")


Kitoboy PIE app loaded


## Inference

In [4]:
def run_kitoboy(texts: list[str], batch_size: int = 32) -> list[list[dict]]:
    predictions = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        payload = {
            "inputs": [
                {
                    "name": "text_input",
                    "shape": [len(batch), 1],
                    "datatype": "BYTES",
                    "data": batch,
                }
            ]
        }
        response = client.post("/v2/models/pie/infer", json=payload)
        response.raise_for_status()
        predictions.extend(response.json()["outputs"][0]["data"])

    return predictions

texts = [item["text"] for item in rows]
predictions = run_kitoboy(texts, batch_size=BATCH_SIZE)
print(f"Predicted rows: {len(predictions)}")

Predicted rows: 1825


## Span Metrics

Считаем exact span match: совпасть должны `start`, `end`, `label`, `text`.

`PASSPORT_RF` исключаем из gold-разметки, потому что старая/текущая система Kitoboy PIE его не поддерживает.

In [5]:
def entity_key(ent: dict) -> tuple:
    return (ent["start"], ent["end"], ent["label"], ent["text"])


def filter_gold_entities(entities: list[dict]) -> list[dict]:
    return [ent for ent in entities if ent["label"] not in EXCLUDED_LABELS]


def compute_span_metrics(rows: list[dict], predictions: list[list[dict]]) -> dict:
    total_gold = 0
    total_pred = 0
    total_tp = 0

    per_label = defaultdict(lambda: Counter({"tp": 0, "fp": 0, "fn": 0}))
    error_rows = []

    for i, (item, pred_entities) in enumerate(zip(rows, predictions)):
        gold_entities = filter_gold_entities(item.get("entities", []))

        gold_set = {entity_key(ent) for ent in gold_entities}
        pred_set = {entity_key(ent) for ent in pred_entities if ent["label"] not in EXCLUDED_LABELS}

        tp_set = gold_set & pred_set
        fp_set = pred_set - gold_set
        fn_set = gold_set - pred_set

        total_gold += len(gold_set)
        total_pred += len(pred_set)
        total_tp += len(tp_set)

        for key in tp_set:
            per_label[key[2]]["tp"] += 1
        for key in fp_set:
            per_label[key[2]]["fp"] += 1
        for key in fn_set:
            per_label[key[2]]["fn"] += 1

        if fp_set or fn_set:
            error_rows.append({
                "row_index": i,
                "jsonl_line": item["_jsonl_line"],
                "text": item["text"],
                "gold_entities": gold_entities,
                "pred_entities": pred_entities,
                "false_positive": sorted(fp_set),
                "false_negative": sorted(fn_set),
            })

    precision = total_tp / total_pred if total_pred else 0.0
    recall = total_tp / total_gold if total_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    label_rows = []
    for label, counts in sorted(per_label.items()):
        tp = counts["tp"]
        fp = counts["fp"]
        fn = counts["fn"]
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        label_rows.append({
            "label": label,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": tp + fn,
        })

    return {
        "micro": {
            "tp": total_tp,
            "fp": total_pred - total_tp,
            "fn": total_gold - total_tp,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "gold_total": total_gold,
            "pred_total": total_pred,
        },
        "per_label": label_rows,
        "error_rows": error_rows,
    }

metrics = compute_span_metrics(rows, predictions)
metrics["micro"]

{'tp': 1173,
 'fp': 1942,
 'fn': 1296,
 'precision': 0.37656500802568216,
 'recall': 0.47509113001215064,
 'f1': 0.4201289398280802,
 'gold_total': 2469,
 'pred_total': 3115}

## Per-Label Metrics

In [6]:
per_label_df = pd.DataFrame(metrics["per_label"])
per_label_df.sort_values("f1")

,label,tp,fp,fn,precision,recall,f1,support
0,ADDRESS,0,528,305,0.000000,0.000000,0.000000,305
7,VK,36,105,273,0.255319,0.116505,0.160000,309
4,ORGANIZATION,134,505,175,0.209703,0.433657,0.282700,309
3,NAME,245,666,64,0.268935,0.792880,0.401639,309
6,TELEGRAM,107,63,201,0.629412,0.347403,0.447699,308
5,PHONE_NUMBER,183,56,126,0.765690,0.592233,0.667883,309
1,BANK_CARD,174,3,136,0.983051,0.561290,0.714579,310
2,EMAIL,294,16,16,0.948387,0.948387,0.948387,310


## Error Inspection

In [7]:
errors = metrics["error_rows"]
print(f"Rows with FP/FN: {len(errors)}")

error_summary_df = pd.DataFrame([
    {
        "error_index": i,
        "jsonl_line": err["jsonl_line"],
        "fp_count": len(err["false_positive"]),
        "fn_count": len(err["false_negative"]),
        "gold_labels": sorted({ent["label"] for ent in err["gold_entities"]}),
        "pred_labels": sorted({ent["label"] for ent in err["pred_entities"]}),
    }
    for i, err in enumerate(errors)
])

error_summary_df.head(50)

Rows with FP/FN: 1342


,error_index,jsonl_line,fp_count,fn_count,gold_labels,pred_labels
0,0,1,1,2,"[PHONE_NUMBER, TELEGRAM]",[TELEGRAM]
1,1,2,1,1,[ORGANIZATION],[ORGANIZATION]
2,2,4,2,2,"[ADDRESS, BANK_CARD]","[ADDRESS, PHONE_NUMBER]"
3,3,5,1,0,[EMAIL],"[EMAIL, NAME]"
4,4,6,2,0,[EMAIL],"[EMAIL, NAME]"
5,5,7,0,1,[VK],[]
6,6,12,1,1,"[ORGANIZATION, TELEGRAM]","[ORGANIZATION, TELEGRAM]"
7,7,13,0,1,"[BANK_CARD, PHONE_NUMBER]",[PHONE_NUMBER]
8,8,14,2,2,"[PHONE_NUMBER, VK]",[NAME]
9,9,15,1,1,"[PHONE_NUMBER, VK]","[PHONE_NUMBER, VK]"


In [8]:
def show_error(error_index: int):
    err = errors[error_index]
    print("=" * 100)
    print(f"Error #{error_index}, jsonl_line={err['jsonl_line']}")
    print("\nTEXT:\n")
    print(err["text"])
    print("\nGOLD:\n")
    print(json.dumps(err["gold_entities"], ensure_ascii=False, indent=2))
    print("\nPRED:\n")
    print(json.dumps(err["pred_entities"], ensure_ascii=False, indent=2))
    print("\nFALSE POSITIVE:\n")
    print(json.dumps(err["false_positive"], ensure_ascii=False, indent=2))
    print("\nFALSE NEGATIVE:\n")
    print(json.dumps(err["false_negative"], ensure_ascii=False, indent=2))

if errors:
    show_error(0)


Error #0, jsonl_line=1

TEXT:

Я учусь в школе. Есть несколько друзей. Там. И лучшая подруга. Ну как лучшая. Для меня лучшая. Насчет ее отношения ко мне не знаю. Но я очень ценю наше общение. Каждая прогулка с ней для меня как праздник. Но, увы, это бывает не часто. Чаще она гуляет с нашими одноклассниками. А влиться в их компанию нн получается. Ну не принимают они меня.

Я даже пытался написать ей в телеграм https://t.me/maria_ivanova_7017, но она так и не ответила. Может, я что-то не так сделал? Или она просто не хочет общаться? Не знаю. Если кто-то из вас её увидит, скажите, что я пытался связаться. Или пусть она сама напишет мне на номер +7 (921) 999-44-66. Мне бы хотя бы понять, в чём дело.

GOLD:

[
  {
    "start": 382,
    "end": 413,
    "label": "TELEGRAM",
    "text": "https://t.me/maria_ivanova_7017"
  },
  {
    "start": 619,
    "end": 637,
    "label": "PHONE_NUMBER",
    "text": "+7 (921) 999-44-66"
  }
]

PRED:

[
  {
    "start": 390,
    "end": 413,
    "label": "TEL

## Label-Level Metrics

Дополнительно считаем мягкую метрику: найден ли тип сущности в тексте хотя бы где-то. Это полезно для сравнения со старым API, который раньше возвращал только labels без границ.

In [9]:
def compute_label_level_metrics(rows: list[dict], predictions: list[list[dict]]) -> dict:
    total_tp = total_gold = total_pred = 0
    per_label = defaultdict(lambda: Counter({"tp": 0, "fp": 0, "fn": 0}))

    for item, pred_entities in zip(rows, predictions):
        gold_labels = {
            ent["label"]
            for ent in item.get("entities", [])
            if ent["label"] not in EXCLUDED_LABELS
        }
        pred_labels = {
            ent["label"]
            for ent in pred_entities
            if ent["label"] not in EXCLUDED_LABELS
        }

        tp = gold_labels & pred_labels
        fp = pred_labels - gold_labels
        fn = gold_labels - pred_labels

        total_tp += len(tp)
        total_gold += len(gold_labels)
        total_pred += len(pred_labels)

        for label in tp:
            per_label[label]["tp"] += 1
        for label in fp:
            per_label[label]["fp"] += 1
        for label in fn:
            per_label[label]["fn"] += 1

    precision = total_tp / total_pred if total_pred else 0.0
    recall = total_tp / total_gold if total_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "micro": {
            "tp": total_tp,
            "fp": total_pred - total_tp,
            "fn": total_gold - total_tp,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "gold_total": total_gold,
            "pred_total": total_pred,
        },
        "per_label": [
            {
                "label": label,
                "tp": counts["tp"],
                "fp": counts["fp"],
                "fn": counts["fn"],
                "precision": counts["tp"] / (counts["tp"] + counts["fp"]) if counts["tp"] + counts["fp"] else 0.0,
                "recall": counts["tp"] / (counts["tp"] + counts["fn"]) if counts["tp"] + counts["fn"] else 0.0,
                "support": counts["tp"] + counts["fn"],
            }
            for label, counts in sorted(per_label.items())
        ],
    }

label_metrics = compute_label_level_metrics(rows, predictions)
label_metrics["micro"]

{'tp': 1751,
 'fp': 789,
 'fn': 717,
 'precision': 0.6893700787401574,
 'recall': 0.7094813614262561,
 'f1': 0.6992811501597443,
 'gold_total': 2468,
 'pred_total': 2540}

In [10]:
pd.DataFrame(label_metrics["per_label"]).sort_values("recall")

,label,tp,fp,fn,precision,recall,support
7,VK,141,0,168,1.000000,0.456311,309
6,TELEGRAM,170,0,138,1.000000,0.551948,308
1,BANK_CARD,177,0,133,1.000000,0.570968,310
5,PHONE_NUMBER,198,40,111,0.831933,0.640777,309
4,ORGANIZATION,209,271,99,0.435417,0.678571,308
0,ADDRESS,247,136,58,0.644909,0.809836,305
3,NAME,299,342,10,0.466459,0.967638,309
2,EMAIL,310,0,0,1.000000,1.000000,310
